# 04 – Tables and data models

nachtkritik.de has been publishing theatre reviews online since 2006. Its archive was exported as one JSON file per entry, more than 40,000 files. In this session we look at what these files contain and how reliable they are, turn them into tables, link two tables, and draw our first charts

**Before we start**

1. Download the two folders for this session from Moodle (100 and 1000 files)
2. Upload them into your Google Drive, so that they end up here:

```
My Drive/
└── review-course/
    └── data/
        └── intern/
            └── 26-nachtkritik/
                ├── cont-jsons-v3-sample-100/
                └── cont-jsons-v3-sample-1000/
```

Please keep the folder names as they are. The files in both folders were picked at random from the whole archive; the 100 files are also part of the 1000

### 1.

We start with a single file

In [ ]:
from pathlib import Path
import json

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/review-course/data")
SAMPLE_100 = DATA_DIR / "intern" / "26-nachtkritik" / "cont-jsons-v3-sample-100"
SAMPLE_1000 = DATA_DIR / "intern" / "26-nachtkritik" / "cont-jsons-v3-sample-1000"

In [ ]:
# json.loads() turns the text of the file into a dictionary, as in the last session
path = SAMPLE_100 / "nk_00007571.json"
record = json.loads(path.read_text(encoding="utf-8"))
print(record.keys())

In [ ]:
# most values are strings
print(record["articleTitle"])
print(record["articleDate"])
print(record["articlePlaintext"][:500])

In [ ]:
# some values are lists: a review can have several critics, a production several authors
print(record["articleAuthors"])
print(record["repAuthors"])

In [ ]:
# repTeam is a list of lists: each inner list is a pair of role and name
print(record["repTeam"])
print(record["repTeam"][0])
print(record["repTeam"][0][1])

Next to the review itself (`article...`), the file describes the production it is about (`rep...`, from "Repertoire"): title, authors, team, actors. This is what makes the archive interesting for us: we can ask not only what the critics wrote, but about which directors, which theatres, which plays

### 2.

Now we read all 100 files of the small sample and check what they contain

In [ ]:
# glob() lists all files in a folder whose name matches a pattern; * stands for any characters
paths = sorted(SAMPLE_100.glob("*.json"))
print(len(paths))
print(paths[:3])

In [ ]:
# a loop reads every file and collects the dictionaries in a list
records = []
for path in paths:
    records.append(json.loads(path.read_text(encoding="utf-8")))
print(len(records))

In [ ]:
# which keys appear, and how often? we count with a dictionary, as in the last session
key_counts = {}
for record in records:
    for key in record:
        if key in key_counts:
            key_counts[key] = key_counts[key] + 1
        else:
            key_counts[key] = 1

for key in key_counts:
    print(key, key_counts[key])

Only a part of the files have a text (`articlePlaintext`). What are the others?

In [ ]:
# we print the first file without a text
for record in records:
    if "articlePlaintext" not in record:
        print(record)
        break

These are not articles at all but pictures that belonged to an article (`ingest_path`). And among the articles, not every one is a review: `convolute_id` sorts the archive into groups

In [ ]:
# the first title of every convolute gives us an idea what the letters stand for
seen = []
for record in records:
    if "articleTitle" in record and record["convolute_id"] not in seen:
        seen.append(record["convolute_id"])
        print(record["convolute_id"], "|", record["articleTitle"][:100])

In [ ]:
# some files also report what went wrong when the archive was exported
for record in records:
    if "errors" in record:
        print(record["errors"])

What we know so far about the quality of the data:

- the export contains pictures and articles, and articles of different kinds; only the convolute `T` seems to contain reviews – but nobody has written down what the letters mean, we can only infer it
- not every field is there in every file: authors, places and the production data are often missing
- some information could not be read during the export (`errors`): a date written as "21 Juni 2013" instead of "21. Juni 2013" was enough

None of this is unusual. Every archive is shaped by the way it was produced, and before we count anything we have to know what we are counting

### 3.

A list of dictionaries is hard to survey. pandas, the most common library for tables in python, turns it into a table: every dictionary becomes a row, every key a column. pandas is already installed in Colab

In [ ]:
# by convention pandas is imported under the short name pd
import pandas as pd

In [ ]:
# from now on we work with the larger sample; reading 1000 files from the Drive can take a minute
records = []
for path in sorted(SAMPLE_1000.glob("*.json")):
    records.append(json.loads(path.read_text(encoding="utf-8")))

# a table in pandas is called a DataFrame
df = pd.DataFrame(records)
print(df.shape)

In [ ]:
# shape gives the number of rows and columns; head() shows the first rows
df.head()

In [ ]:
# a single column is picked by its name, like a value in a dictionary; value_counts() counts its values
df["convolute_id"].value_counts()

In [ ]:
# where a file has no value for a key, the table shows NaN ("not a number", i.e. empty); isna().sum() counts the gaps per column
df.isna().sum()

In [ ]:
# a new column is calculated from an existing one: .str[:4] cuts the first four characters (the year) out of every date
# (the pictures already have a column "year", so we choose a new name)
df["article_year"] = df["dateOfCreation"].str[:4]
df[["dateOfCreation", "article_year"]].head()

Now we keep only the reviews. A condition in square brackets keeps the rows for which it is True

In [ ]:
# first all rows that have a text, then among them those of convolute T
texts = df[df["articlePlaintext"].notna()]
reviews = texts[texts["convolute_id"] == "T"]

# the columns of the pictures are of no use to us; we choose the columns we want to keep with a list of names
reviews = reviews[["article_id", "article_year", "articleTitle", "articleAuthors", "articlePlaces", "repTitle", "repTeam"]]
print(reviews.shape)
reviews.head()

In [ ]:
# even among the reviews, the production data is missing for some
reviews.isna().sum()

### 4.

The column `repTeam` holds a whole list in every cell. With that we cannot easily ask the obvious question: which directors does nachtkritik review most often? Tables work best when every cell holds exactly one value. So we build a second table, with one row for every member of a team

In [ ]:
# in a for-loop over pairs, "role, name" unpacks each pair into two variables
# every row of the new table gets the article_id of its review, so that we can link the tables later
team_rows = []
for record in records:
    if "repTeam" in record and record["convolute_id"] == "T":
        for role, name in record["repTeam"]:
            team_rows.append({"article_id": record["article_id"], "role": role, "name": name})

team = pd.DataFrame(team_rows)
print(team.shape)
team.head(10)

In [ ]:
# which roles are there?
team["role"].value_counts().head(30)

Here is messy data again: "Kostüm" next to "Kostüme", "Choreografie" next to "Choreographie", combinations like "Bühne und Kostüme", and a few entries that went wrong during the export ("ERROR", or a name glued to its role). The archive has recorded what the theatres wrote in their programmes, and they do not follow one standard

In [ ]:
# replace() unifies spellings: the dictionary says which value is replaced by which
team["role"] = team["role"].replace({"Kostüm": "Kostüme", "Choreographie": "Choreografie", "Bühnenbild": "Bühne"})
team["role"].value_counts().head(10)

In [ ]:
# now we can count the directors
directors = team[team["role"] == "Regie"]
directors["name"].value_counts().head(10)

Careful: whoever is listed as "Regie und Bühne" or "Regie, Bühne und Licht" is not counted here. Every decision in cleaning data changes the result, and it is our job to know and report these decisions

Now we have two tables: `reviews` with one row per review and `team` with one row per person and review. They are linked by `article_id`. For this to work, the `article_id` must appear only once in the table of reviews: it is the key of the table

In [ ]:
# is_unique checks whether every value of a column appears only once
print(reviews["article_id"].is_unique)

# in the whole export it does not: the pictures carry the id of the article they belong to
print(df["article_id"].is_unique)

In [ ]:
# merge() links two tables by a common column: every director gets the year and the title of the review
directors_with_reviews = directors.merge(reviews[["article_id", "article_year", "articleTitle"]], on="article_id")
directors_with_reviews.head()

In [ ]:
# the most reviewed director: when and with what?
top = directors["name"].value_counts().index[0]
print(top)
directors_with_reviews[directors_with_reviews["name"] == top]

This is the basic idea of a data model: instead of one big table with lists in its cells, several simple tables, each about one kind of thing (reviews, people), linked by keys. The same could be done for the actors, the critics or the places

In [ ]:
# to_csv() saves a table as a file that Google Sheets or Excel can open; index=False leaves out pandas' row numbers
reviews.to_csv(DATA_DIR / "intern" / "nachtkritik-reviews.csv", index=False)
team.to_csv(DATA_DIR / "intern" / "nachtkritik-team.csv", index=False)

### 5.

pandas can draw simple charts directly from a table

In [ ]:
# how many reviews per year? sort_index() puts the years in order, plot() draws them
reviews["article_year"].value_counts().sort_index().plot(kind="bar", title="Reviews per year (sample of 1000 files)")

In [ ]:
# the ten most frequently reviewed directors; barh draws horizontal bars, which leaves room for the names
directors["name"].value_counts().head(10).sort_values().plot(kind="barh", title="Most reviewed directors")

Our sample contains about 200 reviews, so the bars go up and down by chance: a difference of two or three reviews between two years means nothing. (And 2025 is short because the export was made during that year.) Before we interpret a chart, we have to ask how much data it rests on

### 6. Outlook

Linked tables are what a database stores. Python comes with a small database, SQLite, and pandas can write tables into it. Databases are queried in a language of their own, SQL – we will only take a look here

In [ ]:
import sqlite3

# a database in a single file, here only in Colab's memory; a database cell cannot hold a list, so we only store simple columns
db = sqlite3.connect(":memory:")
reviews[["article_id", "article_year", "articleTitle"]].to_sql("reviews", db, index=False)
team.to_sql("team", db, index=False)
print("tables stored")

In [ ]:
# the same question as above, in SQL: JOIN links the tables, WHERE filters, ORDER BY sorts
query = """
SELECT reviews.article_year, team.name, reviews.articleTitle
FROM team JOIN reviews ON team.article_id = reviews.article_id
WHERE team.role = 'Regie'
ORDER BY reviews.article_year
"""
pd.read_sql(query, db).head(10)

### Exercise

1. Build a third table `critics` with one row per review and author, from the column `articleAuthors` (the loop works like the one for the team). Who wrote the most reviews in our sample?
2. Do the same for `articlePlaces` and draw a chart of the ten most frequent places. What does a place mean here: the city of the theatre, of the festival, of the critic?
3. Add a column `words` to `df` with the number of words of each text (hint: `df["articlePlaintext"].str.split().str.len()`). Are reviews longer than the other kinds of articles?
4. Repeat the chart of reviews per year with the sample of 10,000 files (ask for it in the course; reading takes several minutes). Which years stand out, and why might that be?

To think about: our tables count what the archive has recorded. Which reviews, which theatres, which directors are missing from it, and why?